# Observation Dataset

In [ ]:
from agoutix.cli import ObservationDataset
from pathlib import Path
from rich import print

ANNOTATION_PATH = Path(".").resolve().parent / "observation_positions_dataset.json"
IMAGE_PATH = Path(".").resolve().parent / "assets"

assert ANNOTATION_PATH.exists(), f"Annotation file not found: {ANNOTATION_PATH}"
assert IMAGE_PATH.exists(), f"Image path not found: {IMAGE_PATH}"


In [ ]:
def load_dataset(path: Path | str) -> ObservationDataset:
    """Load an observation dataset from a file.
    If the file contains JSON, use model_validate_json; otherwise fall back to model_validate.
    """
    with open(path, "r", encoding="utf-8") as f:
        return ObservationDataset.model_validate_json(f.read())


dataset = load_dataset(ANNOTATION_PATH)
deployment_ids = set(img.deployment for img in dataset.images)
sequence_ids = set(pos.position_id for pos in dataset.positions)
n_deployments = len(deployment_ids)
n_sequences = len(sequence_ids)
n_observations = len(dataset.observations)
n_positions = len(dataset.positions)
n_images = len(dataset.images)

print(
  f"""[bold green]Dataset loaded successfully![/bold green]
Number of deployments: [bold]{n_deployments}[/bold]
Number of sequences: [bold]{n_sequences}[/bold]
Number of positions: [bold]{n_positions}[/bold]
Number of observations: [bold]{n_observations}[/bold]
Number of images: [bold]{n_images}[/bold]
""")

In [ ]:
positions_by_image_id = {pos.asset_id: pos for pos in dataset.positions}
observations_by_image_id = {img.asset_id: img for img in dataset.images}


def plot_image_positions(image_id: str):
    from matplotlib import pyplot as plt
    from PIL import Image

    img = observations_by_image_id[image_id]
    pos = positions_by_image_id.get(image_id)

    image = Image.open(IMAGE_PATH / img.file_name)

    plt.figure(figsize=(8, 6))
    plt.imshow(image)

    if pos:
        plt.scatter(pos.x, pos.y, c="red", s=10)
        # plt.legend()
    plt.title(f"Image ID: {image_id}")
    plt.axis("off")
    plt.show()


for deployment_id in list(deployment_ids)[:3]:
    print(f"[bold blue]Deployment ID:[/bold blue] {deployment_id}")
    images_in_deployment = [
        img for img in dataset.images if img.deployment == deployment_id
    ]
    for img in images_in_deployment[:2]:
        plot_image_positions(img.asset_id)